# Valuation Engine Prototype

This notebook serves as the experimental environment for the `ValuationEngine`. It demonstrates how to train and evaluate the hybrid model using both financial metrics and NLP embeddings.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_log_error, make_scorer

# Enable autoreload
%load_ext autoreload
%autoreload 2

# Add src to path
sys.path.append(os.path.abspath("../src"))
from valuation import ValuationEngine

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Data Loading
We load the dataset processed by the `FeatureProcessor` (Gold Layer). The engine expects flattened NLP columns (`nlp_0`, `nlp_1`, etc.).

In [2]:
data_path = "../data/processed/PUBLIC_embedded.parquet"

if os.path.exists(data_path):
    df = pd.read_parquet(data_path)
    print(f"Loaded {len(df)} companies with {len(df.columns)} columns.")
    
    # 1. Filter for positive valuations (CRITICAL for log-transform)
    df = df[df['enterprise_value'] > 0]
    print(f"Filtered for positive valuations: {len(df)} companies remaining.")
    
    # 2. Diagnostic Check
    print(f"Diagnostic - Minimum Enterprise Value: {df['enterprise_value'].min():,.2f}")
    display(df.head())
else:
    print(f"Data not found at {data_path}. Creating dummy data for demonstration.")
    # Creating dummy data with flattened NLP features
    n_samples = 100
    df = pd.DataFrame({
        "forwardPE": np.random.uniform(10, 40, n_samples),
        "ev_to_ebitda": np.random.uniform(5, 20, n_samples),
        "ebitda": np.random.uniform(100, 1000, n_samples),
        "total_cash": np.random.uniform(50, 200, n_samples),
        "total_debt": np.random.uniform(10, 100, n_samples),
        "enterprise_value": np.random.uniform(500, 5000, n_samples)
    })
    # Add 384 NLP features
    nlp_data = np.random.rand(n_samples, 384)
    nlp_df = pd.DataFrame(nlp_data, columns=[f"nlp_{i}" for i in range(384)])
    df = pd.concat([df, nlp_df], axis=1)
    display(df.head())


Loaded 8050 companies with 391 columns.
Filtered for positive valuations: 7593 companies remaining.
Diagnostic - Minimum Enterprise Value: 60,814.00


,enterprise_value,forwardPE,ev_to_ebitda,ebitda,total_cash,total_debt,sector,nlp_0,nlp_1,nlp_2,...,nlp_374,nlp_375,nlp_376,nlp_377,nlp_378,nlp_379,nlp_380,nlp_381,nlp_382,nlp_383
0,5.170628e+12,17.027882,31.240,165514002432,5.317200e+10,1.281400e+10,Technology,-0.059150,-0.069233,-0.053639,...,-0.035018,-0.028591,0.014662,-0.092712,0.029643,0.026535,-0.013199,-0.065995,0.014612,0.018574
1,4.609101e+12,26.421965,28.572,161315995648,1.268400e+11,9.587600e+10,Communication Services,-0.059735,-0.116992,0.044080,...,0.011315,-0.044257,-0.024711,-0.135265,0.032019,0.070319,-0.015535,-0.074552,0.032307,-0.036426
2,4.551953e+12,32.155922,28.454,159975997440,6.850700e+10,8.471100e+10,Technology,-0.042323,-0.021482,-0.009602,...,-0.040436,0.024960,0.037074,-0.097955,0.022558,0.120378,0.046419,-0.064816,0.104943,0.013485
3,3.156524e+12,21.645250,17.113,184457003008,7.822800e+10,1.254320e+11,Technology,-0.014270,-0.056247,0.000315,...,0.022693,0.035224,0.073339,-0.102653,0.079596,0.118757,-0.010114,-0.027585,0.031214,-0.044478
4,2.957284e+12,27.011812,18.974,155860992000,1.430890e+11,2.355400e+11,Consumer Cyclical,0.039800,-0.067963,-0.031519,...,0.040109,-0.000710,0.027882,-0.100945,0.072389,0.122982,-0.006967,-0.052227,0.004188,0.016085


## 2. Model Training & Evaluation
We initialize the engine in `public` mode and prepare the data for the default target: `enterprise_value`.

In [3]:
# Initialize engine in public mode
engine = ValuationEngine(mode="public", n_estimators=100)

# Prepare features (X) and target (y)
X, y = engine.prepare_data(df, target_col="enterprise_value")

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {X_train.shape[0]} samples with {X_train.shape[1]} features.")

# Define a small parameter grid for demonstration
param_grid = {
    "preprocess__nlp_prep__n_components": [10, 30, 50],
    "preprocess__fin_prep__imputer__n_neighbors": [3, 5, 10],
    "model__regressor__estimator__max_depth": [3, 5],
    "model__regressor__estimator__learning_rate": [0.1],
    "model__regressor__estimator__n_estimators": [300, 500, 800]
}


# 30, 3, 5, .1, 500 - > .28

# Tune hyperparameters (This replaces engine.train())
best_params = engine.tune_hyperparameters(X_train, y_train, param_distributions=param_grid, n_iter=100, cv=5, scoring = "explained_variance")
print(f"\nBest Parameters Found: {best_params}")

# Evaluate performance using the tuned pipeline
metrics = engine.evaluate(X_test, y_test)

print("\nPerformance Metrics:")
for k, v in metrics.items():
    print(f"{k.upper()}: {v:.4f}")

Training on 6074 samples with 390 features.


Tuning Hyperparameters:   0%|          | 0/1 [00:00<?, ?it/s]

Fitting 5 folds for each of 54 candidates, totalling 270 fits


/Users/ryanfue/Library/Caches/pypoetry/virtualenvs/smartcomps-16436_SS-py3.13/lib/python3.13/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 54 is smaller than n_iter=100. Running 54 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
Tuning Hyperparameters:   0%|          | 0/1 [00:50<?, ?it/s]


ValueError: 
All the 270 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
270 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/ryanfue/Library/Caches/pypoetry/virtualenvs/smartcomps-16436_SS-py3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ryanfue/Library/Caches/pypoetry/virtualenvs/smartcomps-16436_SS-py3.13/lib/python3.13/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/Users/ryanfue/Library/Caches/pypoetry/virtualenvs/smartcomps-16436_SS-py3.13/lib/python3.13/site-packages/sklearn/pipeline.py", line 613, in fit
  

## 3. Visualization
Visualize the Predicted vs Actual Enterprise Values.

In [ ]:
y_pred_df = engine.predict(X_test)
y_pred = y_pred_df["enterprise_value"]

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test["enterprise_value"], y=y_pred, alpha=0.6)
plt.plot([y_test.min().min(), y_test.max().max()], [y_test.min().min(), y_test.max().max()], 'r--', lw=2)
plt.xlabel("Actual Enterprise Value")
plt.ylabel("Predicted Enterprise Value")
plt.title("Actual vs Predicted Valuation")
plt.show()